In [0]:
edf = (
    spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .option("path","/Volumes/workspace/default/practice/Employees.csv")
    .load()
)
display(edf)

In [0]:
edf.write.saveAsTable("tbledf")

In [0]:
endf = (
    spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .option("path","/Volumes/workspace/default/practice/EmployeesNew.csv")
    .load()
)
display(endf)

In [0]:
endf.write.saveAsTable("tblendf")

In [0]:
%sql
select * from tblendf
except
select * from tbledf

In [0]:
%sql
merge into tbledf as t
using tblendf as s
on t.id = s.id
when matched then
update set
t.name = s.name,
t.city = s.city,
t.salary = s.salary
when not matched then
insert(
  id, name, city, salary
) values(
  s.id, s.name, s.city, s.salary
)

In [0]:
%sql
select * from tbledf

In [0]:
%sql
select * from tblendf

In [0]:
from delta.tables import *

In [0]:
employeetable = DeltaTable.forName(spark,"tbledf")
print(type(employeetable))

In [0]:
(
    employeetable.alias("target")
    .merge(source=employeeUpdate_df.alias("Source"),condition="Target.id = Source.id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
%sql
select * from tblemp

In [0]:
(
    employeeTable.alias("Target")
    .merge(source=employeeUpdate_df.alias("Source"),condition="Target.id=Source.id")
    .whenMatchedUpdate((
        set={
            "city":"Source.city",
            "salary":"Source.salary"
        }
    )
     .whenNotMatchedInsert(
         values={
             "id":"Source.id",
             "name":"Source.name",
             "city":"Source.city",
             "salary":"Source.salary"
         }
         ).execute()
     )

In [0]:
%sql
select * from tblemployee